In [1]:
import warnings
warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns


from surprise import accuracy

# class is used to parse a file containing ratings, data should be in structure - user ; item ; rating
from surprise.reader import Reader

# class for loading datasets
from surprise.dataset import Dataset

# for model tuning model hyper-parameters
from surprise.model_selection import GridSearchCV

# for splitting the rating data in train and test dataset
from surprise.model_selection import train_test_split

# for implementing similarity based recommendation system
from surprise.prediction_algorithms.knns import KNNBasic

# for implementing matrix factorization based recommendation system
from surprise.prediction_algorithms.matrix_factorization import SVD

from collections import defaultdict

# for implementing cross validation
from surprise.model_selection import KFold

In [2]:
movies=pd.read_csv('output.csv')

In [17]:
ratings=pd.read_csv('movie_ratings.csv')
ratings.shape

(100004, 5)

In [18]:
movies.head(5)

,adult,genres,movieId,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,runtime,spoken_languages,tagline,title,cast,crew,keywords
0,False,DramaRomance,79782,en,Wenecja,An atmospheric coming-of-age story featuring a...,0.147130,NaN,NaN,2010-05-25,110.0,plPolski,NaN,Venice,Marcin Walewski Magdalena Cielecka Mariusz Bon...,Jan Jakub Kolski Artur Reinhart Witold Chomins...,English
1,False,ComedyHorror,141210,en,The Sleepover,"The town of Derry has a secret, but no one tol...",0.135596,NaN,NaN,2013-10-12,6.0,English,NaN,The Sleepover,Josh Feldman Gus Kamp Carolyn Jania Kristine B...,Chris Cullari,English
2,False,Drama,143750,en,The Farmer's Wife,"As her surroundings are invaded by outsiders, ...",0.211754,NaN,NaN,2012-06-20,18.0,enEnglish,NaN,The Farmer's Wife,James Cartwright Geraldine James Alex Kelly Ma...,John Parish Jonas Grimås Gary Shaw Francis Lee...,short
3,False,Documentary,84198,en,A Place at the Table,"Using personal stories, this powerful document...",0.501046,NaN,USUnitedStatesofAmerica,2012-03-22,84.0,enEnglish,One Nation. Underfed.,A Place at the Table,Jeff Bridges Tom Colicchio Mariana Chilton Ken...,Kristi Jacobson Lori Silverbush,woman director
4,False,Documentary,84198,en,A Place at the Table,"Using personal stories, this powerful document...",0.501046,NaN,USUnitedStatesofAmerica,2012-03-22,84.0,enEnglish,One Nation. Underfed.,A Place at the Table,Jeff Bridges Tom Colicchio Mariana Chilton Ken...,Kristi Jacobson Lori Silverbush,woman director


In [19]:
'''type(movies.loc[0,'movieId'])'''

"type(movies.loc[0,'movieId'])"

In [20]:
#movies.rename(columns={'id':'movieId'},inplace=True)

In [21]:
'''# Replace non-integer values with NaN
movies['movieId'] = pd.to_numeric(movies['movieId'], errors='coerce')

# Convert the column to integer type
movies['movieId'] = movies['movieId'].astype('Int32')'''

"# Replace non-integer values with NaN\nmovies['movieId'] = pd.to_numeric(movies['movieId'], errors='coerce')\n\n# Convert the column to integer type\nmovies['movieId'] = movies['movieId'].astype('Int32')"

In [22]:
movies.shape

(22176, 17)

In [23]:
movies

,adult,genres,movieId,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,runtime,spoken_languages,tagline,title,cast,crew,keywords
0,False,DramaRomance,79782,en,Wenecja,An atmospheric coming-of-age story featuring a...,0.147130,NaN,NaN,2010-05-25,110.0,plPolski,NaN,Venice,Marcin Walewski Magdalena Cielecka Mariusz Bon...,Jan Jakub Kolski Artur Reinhart Witold Chomins...,English
1,False,ComedyHorror,141210,en,The Sleepover,"The town of Derry has a secret, but no one tol...",0.135596,NaN,NaN,2013-10-12,6.0,English,NaN,The Sleepover,Josh Feldman Gus Kamp Carolyn Jania Kristine B...,Chris Cullari,English
2,False,Drama,143750,en,The Farmer's Wife,"As her surroundings are invaded by outsiders, ...",0.211754,NaN,NaN,2012-06-20,18.0,enEnglish,NaN,The Farmer's Wife,James Cartwright Geraldine James Alex Kelly Ma...,John Parish Jonas Grimås Gary Shaw Francis Lee...,short
3,False,Documentary,84198,en,A Place at the Table,"Using personal stories, this powerful document...",0.501046,NaN,USUnitedStatesofAmerica,2012-03-22,84.0,enEnglish,One Nation. Underfed.,A Place at the Table,Jeff Bridges Tom Colicchio Mariana Chilton Ken...,Kristi Jacobson Lori Silverbush,woman director
4,False,Documentary,84198,en,A Place at the Table,"Using personal stories, this powerful document...",0.501046,NaN,USUnitedStatesofAmerica,2012-03-22,84.0,enEnglish,One Nation. Underfed.,A Place at the Table,Jeff Bridges Tom Colicchio Mariana Chilton Ken...,Kristi Jacobson Lori Silverbush,woman director
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22171,False,CrimeDramaThriller,404604,hi,Maa,The bliss of a biology teacher’s family life i...,1.559596,MadFilmsThirdEyePictures,INIndia,2017-07-07,146.0,hiहनद,NaN,Mom,Sridevi Kapoor Sajal Ali Akshaye Khanna Nawazu...,A. R. Rahman Boney Kapoor Mukesh Talreja Sunil...,physical abuse sexual assault
22172,False,ComedyDrama,420346,en,The Morning After,The Morning After is a feature film that consi...,0.139936,OopsDoughnutsProductionsHeandSheFilms,USUnitedStatesofAmerica,2015-01-11,79.0,enEnglish,What happened last night?,The Morning After,Markie Adams Roberto Aguire Tina Arning Lauren...,Shanra J. Kehl Shanra J. Kehl,English
22173,False,Drama,111109,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,0.178241,SineOlivia,PHPhilippines,2011-11-17,360.0,tl,NaN,Century of Birthing,Angel Aquino Perry Dizon Hazel Orencio Joel To...,Lav Diaz Lav Diaz Dante Perez Lav Diaz Lav Dia...,artist play pinoy
22174,False,ActionDramaThriller,67758,en,Betrayal,"When one of her hits goes wrong, a professiona...",0.903007,AmericanWorldPictures,USUnitedStatesofAmerica,2003-08-01,90.0,enEnglish,A deadly game of wits.,Betrayal,Erika Eleniak Adam Baldwin Julie du Page James...,Mark L. Lester C. Courtney Joyner Jeffrey Gold...,English


In [24]:
'''ratings= ratings.drop(['timestamp'], axis=1)'''

"ratings= ratings.drop(['timestamp'], axis=1)"

In [25]:
index=pd.Series(movies.index,index=movies['title']).drop_duplicates()

In [26]:
index

title
Venice                      0
The Sleepover               1
The Farmer's Wife           2
A Place at the Table        3
A Place at the Table        4
                        ...  
Mom                     22171
The Morning After       22172
Century of Birthing     22173
Betrayal                22174
Queerama                22175
Length: 22176, dtype: int64

15 step

In [28]:
ratings.head(5)

,userId,movieId,rating,user_name,email-id
0,1,31,2.5,durbanpadayachee,durbanpadayachee@gmail.com
1,1,1029,3.0,durbanpadayachee,durbanpadayachee@gmail.com
2,1,1061,3.0,durbanpadayachee,durbanpadayachee@gmail.com
3,1,1129,2.0,durbanpadayachee,durbanpadayachee@gmail.com
4,1,1172,4.0,durbanpadayachee,durbanpadayachee@gmail.com


In [13]:
'''
with h5py.File('similarity_genres.h5', 'r') as hf:
    similarity_genres = hf['similarity_genres'][:]


with h5py.File('similarity_overview.h5', 'r') as hf:
    similarity_overview = hf['similarity_overview'][:]

with h5py.File('similarity_language.h5', 'r') as hf:
    similarity_language = hf['similarity_language'][:]

with h5py.File('similarity_credits.h5', 'r') as hf:
    similarity_credits = hf['similarity_credits'][:]
'''

In [29]:
with h5py.File('similarity_genres_list.h5', 'r') as hf:
    similarity_genres_list= hf['similarity_genres_list'][:]


with h5py.File('similarity_overview_list.h5', 'r') as hf:
    similarity_overview_list= hf['similarity_overview_list'][:]

with h5py.File('similarity_language_list.h5', 'r') as hf:
    similarity_language_list= hf['similarity_language_list'][:]

with h5py.File('similarity_credits_list.h5', 'r') as hf:
    similarity_credits_list= hf['similarity_credits_list'][:]

In [30]:
def content_based(title, cosine_sim1=similarity_genres_list,cosine_sim2=similarity_overview_list,cosine_sim3=similarity_language_list,cosine_sim4=similarity_credits_list):

    '''title=mood_data.loc[USERID,mood]'''
    '''index = movies[movies['title']==title].index[0]'''

    idx = index[title]
    
    sorted_similar_movies_1=cosine_sim1[idx][1:5]
    sorted_similar_movies_2=cosine_sim2[idx][1:5]
    sorted_similar_movies_3= cosine_sim3[idx][1:5]
    sorted_similar_movies_4= cosine_sim4[idx][1:5]

    
    sim_idx_1= [i[0] for i in sorted_similar_movies_1]
    sim_idx_2= [i[0] for i in sorted_similar_movies_2]
    sim_idx_3= [i[0] for i in sorted_similar_movies_3]
    sim_idx_4= [i[0] for i in sorted_similar_movies_4]
    
    
    mov_genres = movies[['movieId', 'title','popularity']].iloc[sim_idx_1]
    mov_overview = movies[['movieId', 'title','popularity']].iloc[sim_idx_2]
    mov_language= movies[['movieId', 'title','popularity']].iloc[sim_idx_3]
    mov_credits = movies[['movieId', 'title','popularity']].iloc[sim_idx_4]

    mov=pd.concat([mov_overview, mov_genres,mov_credits,mov_language], axis=0)
    mov.reset_index(drop=True, inplace=True)
    mov.drop_duplicates(inplace=True)

    rec_id=[]
    rec_title=[]
    for i in mov['movieId']:
        rec_id.append(i)

    for i in mov['title']:
        rec_title.append(i)
        
    print(rec_id)
    print(rec_title)
        
    
    return mov

In [32]:
content_based("TMNT")

[388202, 9789, 39286, 1254, 9975, 11619, 1273, 13690, 19598, 48572, 2295, 268092, 84198]
['Arsenal', 'My Brother Is an Only Child', 'Four Last Songs', "Don't Worry, I'm Fine", 'Curious George', 'Flushed Away', 'TMNT', 'Open Season 2', 'Leaving Barstow', 'Red State', 'Clerks II', 'Transformers Prime Beast Hunters: Predacons Rising', 'A Place at the Table']


,movieId,title,popularity
0,388202,Arsenal,4.192948
1,9789,My Brother Is an Only Child,4.391819
2,39286,Four Last Songs,0.550246
3,1254,"Don't Worry, I'm Fine",6.189495
4,9975,Curious George,12.602502
5,11619,Flushed Away,9.922577
6,1273,TMNT,11.358363
7,13690,Open Season 2,9.031276
8,19598,Leaving Barstow,0.134579
9,48572,Red State,6.340655


In [80]:
def train_collaborative(ratings):
    reader = Reader(rating_scale=(0, 5))

    # loading the rating dataset
    data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

    # splitting the data into train and test dataset
    trainset, testset = train_test_split(data, test_size=0.2, random_state=40)
    
    # using SVD matrix factorization
    algo_svd = SVD()

    # training the algorithm on the trainset
    algo_svd.fit(trainset)

    # predicting ratings for the testset
    predictions = algo_svd.test(testset)

    # computing RMSE on the testset
    print(accuracy.rmse(predictions))
    print(accuracy.mae(predictions))

    # set the parameter space to tune
    param_grid = {'n_epochs': [10,20,30], 'lr_all': [0.001, 0.005, 0.01],
              'reg_all': [0.06, 0.08, 0.006]}

    # performing 3-fold gridsearch cross validation
    gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)

    # fitting data
    gs.fit(data)
    para=[]
    para=gs.best_params['rmse']
    # building the optimized SVD model using optimal hyperparameter search
    svd_algo_optimized = SVD(n_epochs=para['n_epochs'], lr_all=para['lr_all'], reg_all=para['reg_all'])

    # training the algorithm on the trainset
    svd_algo_optimized.fit(trainset)

    # predicting ratings for the testset
    predictions = svd_algo_optimized.test(testset)

    # computing RMSE
    print(accuracy.rmse(predictions))
    return svd_algo_optimized

In [81]:
algo=train_collaborative(ratings)

RMSE: 0.8928
0.8927948653788524
MAE:  0.6864
0.6863921831317664
RMSE: 0.8735
0.8735242740375204


33,34--

In [82]:
def get_recommendations(data, user_id, top_n, algo):

    # creating an empty list to store the recommended movie ids
    recommendations = []
    email_ids=[]
    # creating an user item interactions matrix 
    user_item_interactions_matrix = data.pivot(index='userId', columns='movieId', values='rating')
    
    # extracting those movie ids which the user_id has not interacted yet
    non_interacted_movies = user_item_interactions_matrix.loc[user_id][user_item_interactions_matrix.loc[user_id].isnull()].index.tolist()
    
    # looping through each of the movie id which user_id has not interacted yet
    for item_id in non_interacted_movies:
        
        # predicting the ratings for those non interacted movie ids by this user
        est = algo.predict(user_id, item_id).est
        
        # appending the predicted ratings
        recommendations.append((item_id, est))

    # sorting the predicted ratings in descending order
    recommendations.sort(key=lambda x: x[1], reverse=True)
    ids= [i[0] for i in recommendations]
    ids=ids[:top_n]
    existing_ids = set(ids).intersection(set(movies['movieId']))  
    filtered_movies = movies.loc[movies['movieId'].isin(existing_ids)]
    
    return filtered_movies[['movieId','title','popularity']][:top_n]

In [84]:
list_b=get_recommendations(ratings,24, 10, algo)

36--

37--

In [77]:
list_b

,movieId,title,popularity


In [103]:
ratings.tail(5)

,userId,movieId,rating,user_name,email-id
99999,671,6268,2.5,johannesburgmodise,johannesburgmodise@gmail.com
100000,671,6269,4.0,johannesburgmodise,johannesburgmodise@gmail.com
100001,671,6365,4.0,johannesburgmodise,johannesburgmodise@gmail.com
100002,671,6385,2.5,johannesburgmodise,johannesburgmodise@gmail.com
100003,671,6565,3.5,johannesburgmodise,johannesburgmodise@gmail.com


In [113]:
index=ratings.index[ratings['email-id'] == 'johannesburgmodise@gmail.com'].tolist()[0]
user_id = ratings.iloc[index]['userId']

In [114]:
user_id

671

In [118]:
a=list(set(ratings['email-id'].tolist()))

In [119]:
if "aditya@gmail.com" in a:
    print('happy')
else:
    print('sad')

sad
